In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os

sys.path.append(os.path.abspath("../../espaloma-charge_mod/"))
sys.path.append(os.path.abspath('../')) 
import ResParTools as pt
from espaloma_charge import charge
import numpy as np
from rdkit.Chem.Draw import IPythonConsole
import json
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdFMCS
from typing import Callable
# import psiresp
import nglview as nv
import MDAnalysis as mda
import re

import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display, clear_output

IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = (600, 400)

# убрать лишние импорты библиотек, те которые необходимо импротирвоать внутри библиотеки ResParTools

### Логирование

In [ ]:
# Лог для отладки: вызовы функций pt, их параметры, сообщения, словари сопоставления
# и промежуточные молекулы пишутся в logs/<дата_время>/ (log.txt, calls.jsonl, files/).
# Выключить: pt.stop_log()
log_dir = pt.start_log()

In [ ]:
!pwd

# Протокол расчета парциальных зарядов для модифицированных остатков
- На вход подаються .smi-файлы мономера и тримера. 
Для аминокислот: 
- У тримера должны бать группы COO<sup>-</sup> и N<sup>+</sup>H<sub>3</sub> на С и N концах.  

In [ ]:
# Параметры модификации (всё, что отличает одну модификацию от другой)
PTM_name = 'C_AF546'            # имя модификации: файлы шага 4.1 и вход шага 2 ({PTM_name}_rn_H_3D.pdb)
base_name = '546'          # имя остатка в силовом поле (как в rtp)
parent_residue = 'C'  # родительский остаток (однобуквенный код): шаблон, имена, ограничения зарядов
monomer_file = 'molecules/C_AF_546.smiles'    # мономер: SMILES остатка с концевыми группами
trimer_file = 'molecules/ACA_AF_546.smiles'      # тример: остаток с соседями (контекст в цепи)
naming = 'index'                # имена атомов на шаге 3.2: 'greek' (буквенные) или 'index' (большие метки)
resp_run = 'slurm'            # шаг 6 (RESP): 'local' - эта машина, 'slurm' - кластер

## Шаг 1. Открытие мономера и тримера
`pt.file_opener` определяет формат по расширению (`.smiles`, `.pdb`, `.mol2`) и возвращает словарь `{имя файла: молекула RDKit}` с явными водородами.

In [ ]:
monomer_chem = pt.file_opener(monomer_file)
trimers_chem = pt.file_opener([trimer_file])
mon_name = next(iter(monomer_chem))
pol_name = next(iter(trimers_chem))
print(mon_name, pol_name, sep='\n')

## Шаг 2.1 Поиск мономера в тримере
- Поис общей подмолекулы для мономера и тримера
Результат:
- Выделение подмолекулы и дальнейшая работа именно с подмолекулой (основанием)

In [ ]:
match_dict = pt.match_mon_to_pol(monomer_chem, trimers_chem, only_heavy_mapping=True, timeout=5)
sub_chem = match_dict['substructure'][mon_name]
sub_chem_no_H = match_dict['substructure_no_H'][mon_name]
print('атомов в подструктуре:', sub_chem.GetNumAtoms())
print('атомов в подструктуре без водорода:', sub_chem_no_H.GetNumAtoms())

In [ ]:
# мономер и тример рядом, подсвечены сопоставленные атомы
mon_pol = match_dict['mon_pol_matches'][mon_name]  # {индекс в мономере: индекс в тримере}
pt.draw_mol_grid({f'мономер: {mon_name}': monomer_chem[mon_name],
                  f'тример: {pol_name}': trimers_chem[pol_name]},
                 highlight_atoms=[list(mon_pol), list(mon_pol.values())])
# Переход со словарей на свойства молекулы (комментарий в ячейке шага 1) - отдельная задача
# для всей библиотеки; старая draw_mon_pol_match (со словарями) пока работает как раньше.

### 2.2 Сохранение остатка 
- Сохранение молекулы остатка в smi и pdb формате без редактирвоания нумерации атомов и имен атомов

Smiles:
1. Сохраняется молекулярный графф с явными протонами 

PDB: 
1. По умолчанию сохраняется в двухмерном формате (`coords='2D'`).

2. При созранении в трезмерном формате  (`coords='3D'`) :
    -  `stereo={индекс: 'R'/'S'}` - явно заданные стереоцентры;
    - `ca_config='L'` - CA аминокислоты (по умолчанию L, как в белках; L-цистеин по CIP - R, остальные L-аминокислоты - S);
    - `stereo_default='R'` - остальные незаданные центры, функция печатает, какие это центры;
    - `random_seed=42` - одна и та же молекула даёт одни и те же координаты.

In [ ]:
pt.save_chem_to_pdb(sub_chem, f'molecules/substructure/3_{mon_name}')
pt.save_chem_to_smiles(sub_chem, f'molecules/substructure/3_{mon_name}', canonical=False, atom_map=True)
pt.save_chem_to_pdb(sub_chem_no_H, f'molecules/substructure/3_{mon_name}_no_H')
pt.save_chem_to_smiles(sub_chem_no_H, f'molecules/substructure/3_{mon_name}_no_H', canonical=False, atom_map=True)
# save_aa_chem_to_smiles объединена с save_chem_to_smiles; старое имя работает (для старых ноутбуков)

## Шаг 3 Переиндексация атомов и переименовка атомов
- Меняется порадок записи атомов их индексы и имена для воспоизведения номенклатуры Gromacs

Шаг выполняется двумя ячейками:
1. **Проверка** - файлы шага 2.2 читаются обратно, шаги 3.1 и 3.2 выполняются в памяти, на одной картинке три этапа: исходная молекула (индексы), после перенумерации (индексы), с именами атомов. Подсвечен родительский остаток. Файлы не пишутся.
2. **Сохранение** - если картинка верна, пишутся `3_1_<имя>.pdb` и `3_2_<имя>.pdb` (+ `_no_H`). Параметры остатка задаются один раз (`residue_params`) и используются в обеих ячейках.

### Шаг 3.1
В модифицированном остатке часть атомов от родительского остатка; они получают номера в порядке шаблона (`N H CA HA CB HB1 HB2 СG ... C O`). Остальные атомы нумеруются обходом молекулярного графа: новые атомы встают сразу за каноническим атомом, к которому пришиты, каждый водород - за своим тяжёлым атомом. Порядок не зависит от исходной нумерации и от стереохимии. На картинке подсвечены атомы родительского остатка (`ParentAtoms`).

Результат сохраняется в `3_1_<имя>.pdb`. 

## Шаг 3.2. Имена атомов и сохранение остатка
- Атомам присваиваются канонические имена в составе остатка. 
- Задается имя остатка (не более 3 символов), номер и цепь. 
- Сохраняются версии с водородами и без (`_no_H`).
- Остов всегда `N CA C O`, водород при N - `H`, при CA - `HA`.

#### Две реализации системы присовения имен атомов: 
**Способ 1 - буквенный, `naming='greek'` (по умолчанию).** Как в amber14sb. Тяжёлые атомы боковой цепи делятся на уровни по числу связей от CA, уровень задаёт букву: 1 - B, 2 - G, 3 - D, 4 - E, 5 - Z, 6 - H, 7 - T, 8 - I, 9 - K, 10 - L, 11 - M, 12 - N. Номер добавляется, только если на уровне несколько тяжёлых атомов. Водород = `H` + имя своего атома без элемента (+ номер, если водородов несколько).
```
уровень:     0    1     2     3     4     5     6
             CA - CB  - CG  - CD  - CE  - NZ  - CH1   HH11 HH12 HH13
             |    HB1   HG1   HD1   HE1   |
             HA   HB2   HG2   HD2   HE2   + - CH2   HH21 HH22 HH23
                                          |
                                          + - CH3   HH31 HH32 HH33
                                    (Lysine_3M: на уровне 6 три атома -> CH1, CH2, CH3)
```
Подходит для небольших остатков: букв хватает на 12 уровней, имя - не больше 4 символов. Если длина радикала больше 12, то функция предупреждает и переключается на **способ 2**.

**Способ 2 - по индексу, `naming='index'`.** Атомы родительского остатка (`parent_atoms`) называются буквенным способом, все остальные - элемент + индекс атома из.
```
индекс:      0 1  2  3   4   5   6    7    8    9     10
             N-H  CA-HA  CB  HB1 HB2  SG - C8 - H9    C10 ...   метка: элемент + индекс
                  |      |            |
                  +------+------------+   родительский остаток (подсвечен на шаге 3.1):
                                          буквенные имена N H CA HA CB HB1 HB2 SG ... C O
```

In [ ]:
# проверка: файлы шага 2.2 читаются обратно, число атомов совпадает
pdb_chem = pt.file_opener(f'molecules/substructure/3_{mon_name}.pdb')[f'3_{mon_name}']
smiles_chem = pt.file_opener(f'molecules/substructure/3_{mon_name}.smiles')[f'3_{mon_name}']  # file_opener возвращает словарь
print('атомов: pdb', pdb_chem.GetNumAtoms(), '| smiles', smiles_chem.GetNumAtoms(), '| подструктура', sub_chem.GetNumAtoms())

# 3.1 перенумерация; parent_atoms - новые индексы атомов родительского остатка
renum_chem = pt.renumber_residue_atoms(pdb_chem, ref_base_name=parent_residue, timeout=5)
parent_atoms = json.loads(renum_chem.GetProp('ParentAtoms'))
old_indices = json.loads(renum_chem.GetProp('OldIndices'))  # на позиции нового индекса - старый

# 3.2 имена атомов и параметры остатка (в памяти, на копии)
residue_params = dict(resname=base_name, resid=2, segid='A', naming=naming, parent_atoms=parent_atoms)
named_chem = pt.rdkit_pdb_modification(Chem.Mol(renum_chem), **residue_params)

pt.draw_mol_grid([pdb_chem, renum_chem, named_chem],
                 legends=['2.2: исходные индексы', '3.1: перенумерация', '3.2: имена атомов'],
                 highlight_atoms=[[old_indices[i] for i in parent_atoms], parent_atoms, parent_atoms],
                 show_index=[True, True, False], show_names=[False, False, True], sub_img_size=(700,900))

In [ ]:
# сохранение шагов 3.1 и 3.2 (после проверки картинки)
pt.save_chem_to_pdb(renum_chem, f'molecules/substructure/3_1_{mon_name}')
pt.save_aa_chem_to_pdb(renum_chem, f'molecules/substructure/3_2_{mon_name}', **residue_params)

## Шаг 4. Проверка и правка остатка
Вход - файл шага 3.2 (`3_2_<имя>.pdb`), выход - `4_<имя>.pdb` и `general_atoms.json`.
1. **Проверка** - остаток читается из файла: число атомов, имя, номер и цепь остатка, картинка с именами атомов (подсвечен родительский остаток).
2. **Правка (при необходимости)** - переименование атомов и смена параметров остатка.
3. **Дубликаты** - имена атомов в остатке должны быть уникальными, иначе шаг останавливается.
4. **Сохранение** - `4_<имя>.pdb` и `general_atoms.json`: имена атомов родительского остатка, по ним `3_edd_topology.ipynb` берёт типы атомов из родительской аминокислоты.

In [ ]:
# остаток из шага 3.2: имена атомов, имя и номер остатка, цепь
residue_chem = pt.file_opener(f'molecules/substructure/3_2_{mon_name}.pdb')[f'3_2_{mon_name}']
res_params = {(a.GetPDBResidueInfo().GetResidueName(), a.GetPDBResidueInfo().GetResidueNumber(),
               a.GetPDBResidueInfo().GetChainId()) for a in residue_chem.GetAtoms()}
print('атомов:', residue_chem.GetNumAtoms(), '| остаток (имя, номер, цепь):', res_params)
# подписи - имена атомов, подсвечен родительский остаток
pt.draw_molecule(residue_chem, highlight_atoms=parent_atoms, show_names=True, show_index=False)

### Изменение имён атомов и параметров остатка (при необходимости)
`rename_map = {'старое имя': 'новое имя'}`; если новое имя уже занято другим атомом, тот атом получает следующее свободное имя (`CB` → `CB1`). Пустой словарь - имена не меняются, задаются только имя, номер и цепь остатка.

In [ ]:
rename_map = {}
residue_chem = pt.modifie_residue_info(residue_chem, rename_map, resname=base_name, resid=2, segid='A')

### Проверка имён атомов на дубликаты

In [ ]:
duplicates = pt.check_duplicate_atom_names(residue_chem)
if duplicates:
    raise ValueError(f'Повторяющиеся имена атомов (имя: индексы): {duplicates}')

### Сохранение результата шага 4

In [ ]:
pt.save_chem_to_pdb(residue_chem, f'molecules/substructure/4_{mon_name}')
# имена атомов родительского остатка - для 3_edd_topology.ipynb
general_atoms = [residue_chem.GetAtomWithIdx(i).GetProp('AtomName') for i in parent_atoms]
print('general_atoms:', general_atoms)
pt.save_json('general_atoms', general_atoms)

### Шаг 4.1. Протонирование атомов, потерявших стандартную валентность 

Этот шаг необходим для дальнейшего получения файла itp для нашей а.к. \
Дело в том, что acpype отказывается создавать топологию для атомов с нестандартными валентностями, что вполне логично. Поэтому мы добавляем протоны, которых на самом деле нет в остатке. После в процесе редактирвоания itp мы удалим все строки с лишними протонами. \
Стоит отметить, что этот шаг можно также выполнить с использованием любого удобного вам молекулярного редактора (ChimeraX, PyMOL) взяв за основу pdb фаийл сохраненный на 4-ом шаге.

Как работает `pt.add_protons_and_renumber_H`:
- водороды достраиваются у атомов с неполной валентностью: у вырезанного из тримера остатка это N (вместо связи с предыдущим остатком) и C (вместо связи со следующим); атомы с формальным зарядом (O<sup>-</sup> сульфогрупп) не протонируются, заряд остатка не меняется;
- атомы остатка сохраняют индексы и имена, новые водороды добавляются в конец и получают имена `HW1`, `HW2`, ...;
- индексы новых водородов - в свойстве молекулы `AddedH` (на картинке подсвечены).

Сохраняются:
- `4_1_<имя>.pdb` / `.smiles` - результат шага (2D, как остальные этапы);
- `<PTM_name>_rn_H_3D.pdb` - вход шага 2 (`2_generate_topology.ipynb`, acpype): 3D-структура, CA - L, остальные стереоцентры - R (см. шаг 2.2 и «Дополнительные инструменты»).

In [ ]:
H_residue = pt.add_protons_and_renumber_H(residue_chem, resname=base_name, resid=2, segid='A')
added_H = json.loads(H_residue.GetProp('AddedH'))
print('атомов:', H_residue.GetNumAtoms(), '| заряд остатка:', Chem.GetFormalCharge(H_residue))
pt.draw_molecule(H_residue, highlight_atoms=added_H, show_names=True, show_index=False)

In [ ]:
pt.save_chem_to_pdb(H_residue, f'molecules/substructure/4_1_{mon_name}')
pt.save_chem_to_smiles(H_residue, f'molecules/substructure/4_1_{mon_name}', canonical=False, atom_map=True)
# вход шага 2 (acpype): 3D-структура с заданной стереохимией
pt.save_chem_to_pdb(H_residue, f'molecules/substructure/{PTM_name}_rn_H_3D', coords='3D')

In [ ]:
# 3D-вход acpype: проверка геометрии
nv.show_mdanalysis(mda.Universe(f'molecules/substructure/{PTM_name}_rn_H_3D.pdb'))

## Дополнительные инструменты
Не обязательны для пайплайна: для проверки промежуточных результатов и нестандартных случаев.

- **`pt.draw_molecule(mol, show_index=True, show_names=False, highlight_atoms=None, charge_list=None)`** - рисунок молекулы с подписями «индекс:имя: заряд». 
Зачем: проверить нумерацию, имена атомов, заряды; подсветить группу атомов (родительский остаток, добавленные водороды). Старое имя `draw_mol_with_atom_index` работает.
- **`pt.draw_mol_grid(mols, legends, highlight_atoms, show_index, show_names, charge_lists)`** - несколько молекул на одной картинке; параметры - одно значение на все или список по молекулам. Зачем: сравнить этапы (как на шаге 3) или мономер и тример.
- **`nv.show_mdanalysis(mda.Universe(путь))`** - 3D-просмотр PDB. Зачем: геометрия 3D-файлов (вход acpype); 2D-файлы этапов в нём плоские.
- **Таблица атомов** - индекс, имя, остаток, заряд. Зачем: найти атом по имени или индексу для `rename_map` (шаг 4) или ограничений зарядов (шаг 5).
- **`pt.save_chem_to_pdb(mol, путь, coords='3D', stereo={индекс: 'R'/'S'}, ca_config='L', stereo_default='R', random_seed=42)`** - 3D-структура с заданной стереохимией (`pt.set_mol_coords` - то же без записи файла). Зачем: вход acpype, другая конфигурация стереоцентров метки, D-аминокислота (`ca_config='D'`).
- **`pt.modifie_residue_info(mol, rename_map, resname, resid, segid)`** и **`pt.check_duplicate_atom_names(mol)`** - правка имён и параметров остатка и проверка уникальности имён (используются на шаге 4, работают с любой молекулой с именами атомов).
- **Лог** - `pt.start_log()` (ячейка в начале ноутбука), `pt.log_note('текст', данные=...)`, `pt.stop_log()`. Зачем: подробности вызовов, словари сопоставления и промежуточные молекулы в `logs/<дата_время>/` без засорения вывода ячеек.

In [ ]:
# рисунок: только имена атомов, без индексов
pt.draw_molecule(residue_chem, show_names=True, show_index=False)

In [ ]:
# таблица атомов: индекс, имя, остаток, формальный заряд
for atom in H_residue.GetAtoms():
    info = atom.GetPDBResidueInfo()
    print(atom.GetIdx(), atom.GetProp('AtomName'), info.GetResidueName(), info.GetResidueNumber(),
          info.GetChainId(), atom.GetFormalCharge())

In [ ]:
# 3D-структура (пример, файл не пишется): CA - L, остальные стереоцентры - R.
# Свою конфигурацию задают так: stereo={индекс: 'R' или 'S'} (индексы стереоцентров печатает функция)
mol_3d = pt.set_mol_coords(Chem.Mol(H_residue), coords='3D', ca_config='L', stereo_default='R')
nv.show_rdkit(mol_3d)

## Шаг 5. Расчет зарядов при помощи библиотеки Espaloma Charge
**Вход** - остаток шага 4 (`4_<имя>.pdb`): без добавленных водородов HW шага 4.1, атомы в том же порядке, что в топологии после удаления HW в `3_edd_topology.ipynb` - заряды сопоставляются атомам по порядку.

У вырезанного остатка N и C без соседних остатков; Espaloma видит их валентность через признаки атомов (число соседей с неявными водородами), а заряды считает только для атомов молекулы.

**Ограничения** - атомы остова `N H CA HA C O` получают заряды одноимённых атомов родительского остатка из amber14sb (`aminoacids.rtp`), остальным атомам заряды считает Espaloma так, чтобы сумма была равна заряду остатка (сумма формальных зарядов атомов, здесь -2).

**Выход** - `AI_chrges_<имя>.json`: заряды в порядке атомов остатка шага 4.

In [ ]:
residue_chem = pt.file_opener(f'molecules/substructure/4_{mon_name}.pdb')[f'4_{mon_name}']
print('атомов:', residue_chem.GetNumAtoms(), '| заряд остатка:', Chem.GetFormalCharge(residue_chem))
pt.draw_molecule(residue_chem, highlight_atoms=parent_atoms, show_names=True, show_index=False)

### Ограничения зарядов
`pt.charge_constraints_from_rtp(mol, rtp_residue, atom_names=('N','H','CA','HA','C','O'))` - `{индекс: заряд}` по именам атомов; остаток в rtp задаётся явно (для гистидина в amber - `HID`/`HIE`/`HIP`, для цистеина с дисульфидом - `CYX`).

In [ ]:
rtp_residue = pt.AMINO_ACIDS[parent_residue][0].upper()  # например 'CYS' или 'LYS'
constraints = pt.charge_constraints_from_rtp(residue_chem, rtp_residue)

### Расчет зарядов при помощи библиотеки Espaloma Charge

In [ ]:
charges = charge(residue_chem, constraints=constraints)
pt.log_note('заряды Espaloma', constraints=constraints, charges=charges.tolist())

# проверки: сумма равна заряду остатка, ограниченные атомы получили заданные заряды
total = Chem.GetFormalCharge(residue_chem)
fixed_ok = all(abs(charges[i] - q) < 1e-5 for i, q in constraints.items())
print(f'сумма зарядов: {charges.sum():.5f} (заряд остатка {total}) | ограничения соблюдены: {fixed_ok}')
if abs(charges.sum() - total) > 1e-3 or not fixed_ok or len(charges) != residue_chem.GetNumAtoms():
    raise ValueError('Заряды не прошли проверку: см. сумму и ограничения выше.')

In [ ]:
pt.draw_molecule(residue_chem, charge_list=charges, show_names=False, show_index=False)

## Сохранение масcива зарядов

In [ ]:
pt.save_charges_json(name=f'AI_chrges_{mon_name}', charge_list=charges.tolist())

## Шаг 6. Расчёт зарядов RESP (альтернатива шагу 5)
Входные данные те же, что у Espaloma: остаток шага 4 (`residue_chem`) и ограничения остова (`constraints`). Протокол - как для зарядов amber ff94/ff14SB/ff19SB (Cornell 1995):
- **молекула** - остаток с кэпами ACE и NME (дипептид ACE-X-NME); заряды кэпов фиксированы на значениях ACE/NME amber14sb;
- **конформации** - пары углов остова (φ, ψ) × `n_sidechain` конформаций боковой цепи (из пула RDKit, MMFF с замороженными φ/ψ, низкоэнергетические и непохожие, без контакта боковой цепи с остовом);
- **оптимизация** HF/6-31G* в psi4 с замороженными φ/ψ, затем ESP (сетка MSK) и двухстадийный RESP; все конформации с равным весом.

`fix_backbone`:
- `True` (по умолчанию) - αR (−60°, −40°) и β (−120°, 130°);
- `False` - без фиксации: углы остова получаются при генерации конформаций и оптимизации;
- свои пары: `[(-60, -40), (-75, 145)]` или `{'alphaR': (-60, -40), 'PPII': (-75, 145)}`.

Где считать (`run`):
- `'local'` - на этой машине: расчёт идёт в окружении `darwin_resp` отдельным процессом (ядро ноутбука остаётся `darwin_ec`), `n_threads` - по числу физических ядер;
- `'slurm'` - готовит `run_resp.sbatch` в папке задачи (по умолчанию 24 ядра, 32 ГБ, 8 часов; конформации считаются параллельно). Отправка: `cd <папка> && sbatch run_resp.sbatch`; если задачу оборвёт лимит времени, повторная отправка продолжит с сохранённых геометрий.

Для больших остатков (метки, около 100 атомов и больше) одна конформация считается десятки часов даже на 24 ядрах - для них `resp_run='slurm'`. Файлы задачи - `RESP_data/<имя>/` (в git не попадают).

In [ ]:
# задача RESP: молекула ACE-остаток-NME, конформации, ограничения (без квантовой химии)
# стереохимия (CA - L, центры боковой цепи, двойные связи) берётся из мономера шага 1
resp_folder = pt.prepare_resp_job(residue_chem, constraints, f'{mon_name}_capped',
                                  fix_backbone=True, n_sidechain=3, monomer=monomer_chem[mon_name])

In [ ]:
# запуск: resp_run из ячейки параметров ('local' - считать здесь, 'slurm' - задача для кластера)
pt.run_resp(resp_folder, run=resp_run, n_threads=4, n_parallel=1,
            slurm=dict(cpus=24, mem='32G', time_limit='08:00:00'))

### Результат RESP и сравнение с Espaloma
Заряды читаются из папки задачи (после расчёта на этой машине или на кластере) и проверяются: сумма равна заряду остатка, ограничения остова соблюдены. Сравнение со схемой Espaloma - по группам (тяжёлый атом + его водороды): AM1-BCC и RESP по-разному делят заряд внутри группы, а для электростатики важен заряд групп.

In [ ]:
resp_q = pt.load_resp_result(resp_folder)
pt.compare_charges(residue_chem, {'Espaloma': charges, 'RESP': resp_q})
pt.draw_molecule(residue_chem, charge_list=resp_q, show_names=False, show_index=False)

In [ ]:
pt.save_charges_json(name=f'RESP_chrges_{mon_name}', charge_list=[float(q) for q in resp_q])